# MMGMT 721 Milestone 2 – Data wrangling pipeline

**Project:** USD/CAD exchange rate and Canadian energy exports to the U.S. · Study period: January 2018 – December 2025

This notebook is shared by the group. Each part starts with a heading **Part N – topic (name)**. Add your code and notes under your own part and run the notebook from the top (**Run All**) so that later parts can use the results of earlier parts.

| Part | Content | Owner |
|---|---|---|
| Part 2 | Data quality checks and cleaning | Rui |
| Part 3 | Data transformation and integration |    Daisy |
| Part 4 | Prepared dataset and data storage | Emily|


The data dictionary (Part 1) is kept in `docs/`.

## Part 2 – Data quality checks and cleaning (Rui)


**Input**
- Statistics Canada Table 12-10-0182-01 (monthly exports, thousands of CAD)
- Bank of Canada daily exchange rate FXUSDCAD (CAD per 1 USD)

**Output**
- `data/clean/exports_clean.csv` and `data/clean/fx_daily_clean.csv` (optionally also saved to MotherDuck)
- `data/clean/quality_log.csv` and `docs/cleaning_log.md`, which are filled in automatically

**How to use:** check the settings in the next cell, then click **Run All**. Each check prints its result, and the cleaning log at the end is built from those results.

In [1]:
# ===================== SETTINGS (edit here) =====================
SOURCE = "csv"              # "csv" = read files from data/raw ; "motherduck" = read raw tables from MotherDuck
SAVE_TO_MOTHERDUCK = False  # True = also write the clean tables to MotherDuck (schema "clean")

# Raw files in data/raw/ (None = find automatically by name)
EXPORTS_CSV = None          # e.g. "12100182.csv"
FX_CSV = None               # e.g. "FXUSDCAD.csv"

# MotherDuck (only used when SOURCE = "motherduck" or SAVE_TO_MOTHERDUCK = True)
MD_DATABASE = "energy_fx"
RAW_EXPORTS_TABLE = "raw.exports"
RAW_FX_TABLE = "raw.fx_daily"

# Study period
START_MONTH = "2018-01"
END_MONTH = "2025-12"

# Which StatCan series to keep (matched case-insensitively)
TRADE_KEEP = "Export"              # StatCan label for total exports in this table
PRODUCT_CODE_PATTERN = r"\[(?:C12|14[1-6]|261)\]"   # Energy products [C12] and its 7 sub-groups [141]–[146], [261]
DEST_US = "United States"
DEST_ALL = "All countries"    # optional: only used if "All countries" was downloaded
ORIGIN_KEEP = "Canada"       # country of origin: Canada = goods produced in Canada

# Thresholds used to flag (not delete) unusual values
EXPORT_OUTLIER_PCT = 0.25   # month-over-month change in export value larger than ±25%
FX_JUMP_PCT = 0.02          # day-over-day change in the exchange rate larger than ±2%

In [2]:
from pathlib import Path
import io
import re

import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR, CLEAN_DIR, DOCS_DIR = ROOT / "data" / "raw", ROOT / "data" / "clean", ROOT / "docs"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(ROOT / ".env")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

PERIOD_START = pd.Timestamp(START_MONTH + "-01")
PERIOD_END = pd.Timestamp(END_MONTH + "-01") + pd.offsets.MonthEnd(0)
N_MONTHS = len(pd.period_range(START_MONTH, END_MONTH, freq="M"))

LOG = []
def log(dataset, check, finding, decision, rationale):
    """Record one row of the cleaning log and print the finding."""
    LOG.append({"Dataset": dataset, "Check": check, "Finding": finding,
                "Decision": decision, "Rationale": rationale})
    print(f"[{dataset}] {check}: {finding}")

def find_file(patterns):
    for pattern in patterns:
        hits = sorted(RAW_DIR.glob(pattern))
        if hits:
            return hits[0]
    files = [f.name for f in RAW_DIR.iterdir() if f.name != ".gitkeep"]
    raise FileNotFoundError(f"No file matching {patterns} in {RAW_DIR}. Files found: {files}")

def find_col(df, *keywords):
    for c in df.columns:
        if all(k.lower() in c.lower() for k in keywords):
            return c
    return None

_con = None
def md_con():
    """Connect to MotherDuck (reads the motherduck_token from .env)."""
    global _con
    if _con is None:
        _con = duckdb.connect(f"md:{MD_DATABASE}")
    return _con

print("Project folder:", ROOT)
print(f"Study period: {START_MONTH} to {END_MONTH} ({N_MONTHS} months)")

Project folder: c:\Users\chenr\Projects\721\energy-fx
Study period: 2018-01 to 2025-12 (96 months)


### 2.1 Energy exports – Statistics Canada Table 12-10-0182-01

In [3]:
if SOURCE == "csv":
    exp_path = RAW_DIR / EXPORTS_CSV if EXPORTS_CSV else find_file(["*12100182*.csv", "*1210018201*.csv"])
    raw_exp = pd.read_csv(exp_path, dtype=str, encoding="utf-8-sig")
    print("Read file:", exp_path.name)
else:
    raw_exp = md_con().sql(f"SELECT * FROM {RAW_EXPORTS_TABLE}").df()
    print("Read MotherDuck table:", RAW_EXPORTS_TABLE)

raw_exp.columns = [c.strip() for c in raw_exp.columns]
print("Rows, columns:", raw_exp.shape)
raw_exp.head()

Read file: 1210018201_databaseLoadingData.csv
Rows, columns: (768, 18)


,REF_DATE,GEO,DGUID,North American Product Classification System (NAPCS),Trade,Principal trading partner: Exports by country of destination,Principal trading partner: Exports by country of origin,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2018-01,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,10020138,NaN,NaN,NaN,0
1,2018-02,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,7772901,NaN,NaN,NaN,0
2,2018-03,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,8436578,NaN,NaN,NaN,0
3,2018-04,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,8343371,NaN,NaN,NaN,0
4,2018-05,Canada,2021A000011124,Energy products [C12],Export,"United States, country of destination","Canada, country of origin",Dollars,81,thousands,3,v1863565574,1.12.1.2.2,9491938,NaN,NaN,NaN,0


In [4]:
# Identify the dimension columns (names can differ slightly between downloads)
COL_TRADE = "Trade" if "Trade" in raw_exp.columns else find_col(raw_exp, "trade")
COL_PRODUCT = find_col(raw_exp, "napcs") or find_col(raw_exp, "product")
COL_DEST = find_col(raw_exp, "destination")
COL_ORIGIN = find_col(raw_exp, "origin")
for name, col in [("Trade", COL_TRADE), ("Product", COL_PRODUCT), ("Destination", COL_DEST), ("Origin", COL_ORIGIN)]:
    print(f"{name:12s} -> {col}")
assert COL_TRADE and COL_PRODUCT and COL_DEST, "A key column was not found: check the column names printed above."

# Show the categories available, so the filters in SETTINGS can be checked
for col in [COL_TRADE, COL_DEST, COL_ORIGIN]:
    if col:
        print(f"\n{col}:", sorted(raw_exp[col].dropna().astype(str).unique())[:40])
energy_products = sorted(v for v in raw_exp[COL_PRODUCT].dropna().astype(str).unique() if re.search(PRODUCT_CODE_PATTERN, v))
print(f"\nEnergy product categories found ({len(energy_products)}):", energy_products)

Trade        -> Trade
Product      -> North American Product Classification System (NAPCS)
Destination  -> Principal trading partner: Exports by country of destination
Origin       -> Principal trading partner: Exports by country of origin

Trade: ['Export']

Principal trading partner: Exports by country of destination: ['United States, country of destination']

Principal trading partner: Exports by country of origin: ['Canada, country of origin']

Energy product categories found (8): ['Coal [144]', 'Crude oil and bitumen [141]', 'Electricity [146]', 'Energy products [C12]', 'Natural gas [142]', 'Natural gas liquids (including condensate) and related products [143]', 'Nuclear fuel and other energy products [145]', 'Refined petroleum energy products (including liquid biofuels) [261]']


#### 2.1.1 Data types

In [5]:
exp = raw_exp.copy()
exp["month"] = pd.to_datetime(exp["REF_DATE"].astype(str).str[:7], format="%Y-%m", errors="coerce")
exp["value"] = pd.to_numeric(exp["VALUE"], errors="coerce")

bad_dates = int(exp["month"].isna().sum())
print(raw_exp[["REF_DATE", "VALUE"]].dtypes)
log("Exports", "Data types",
    f"REF_DATE and VALUE were read as {raw_exp['REF_DATE'].dtype} / {raw_exp['VALUE'].dtype}; {bad_dates} dates could not be parsed",
    "Converted REF_DATE to a monthly date (first day of month) and VALUE to a number",
    "Dates and numbers need the correct type for sorting, joining with the exchange-rate data and calculating changes")

REF_DATE    str
VALUE       str
dtype: object
[Exports] Data types: REF_DATE and VALUE were read as str / str; 0 dates could not be parsed


#### 2.1.2 Keep only the series needed for the research question

In [6]:
n_raw = len(exp)
in_period = exp["month"].between(PERIOD_START, PERIOD_END)
is_trade = exp[COL_TRADE].astype(str).str.strip().str.lower() == TRADE_KEEP.lower()
is_product = exp[COL_PRODUCT].astype(str).str.contains(PRODUCT_CODE_PATTERN, regex=True, na=False)
# StatCan labels look like "United States, country of destination": compare only the part before the comma
dest_short = exp[COL_DEST].astype(str).str.split(",").str[0].str.strip()
is_dest = dest_short.str.lower().isin([DEST_US.lower(), DEST_ALL.lower()])

for label, m in [("trade type", is_trade), ("energy products", is_product), ("destination", is_dest), ("study period", in_period)]:
    if not m.any():
        raise ValueError(f"No rows match the {label} filter. Check the SETTINGS cell against the categories printed above.")

mask = in_period & is_trade & is_product & is_dest
if COL_ORIGIN:
    origin_short = exp[COL_ORIGIN].astype(str).str.split(",").str[0].str.strip()
    is_origin = origin_short.str.lower() == ORIGIN_KEEP.lower()
    if not is_origin.any():
        raise ValueError(f"No rows match ORIGIN_KEEP = {ORIGIN_KEEP!r}. Options: {sorted(origin_short.unique())}")
    mask &= is_origin
    print("Country of origin kept:", ORIGIN_KEEP)

dropped_trade = sorted(set(exp.loc[in_period & is_product & is_dest & ~is_trade, COL_TRADE].astype(str)))
exp = exp[mask].copy()
exp["product_code"] = exp[COL_PRODUCT].astype(str).str.extract(r"\[([^\]]+)\]\s*$", expand=False)
exp["product_name"] = exp[COL_PRODUCT].astype(str).str.replace(r"\s*\[.*\]\s*$", "", regex=True)
exp["destination"] = exp[COL_DEST].astype(str).str.split(",").str[0].str.strip()

log("Exports", "Series selection",
    f"{n_raw:,} raw rows reduced to {len(exp):,} rows; trade types dropped: {dropped_trade or 'none'}",
    f"Kept exports (Trade = '{TRADE_KEEP}') of Energy products [C12] and its {exp['product_code'].nunique() - 1} sub-groups, "
    f"country of origin = {ORIGIN_KEEP}, destination = {', '.join(sorted(exp['destination'].unique()))}, {START_MONTH} to {END_MONTH}",
    "Canadian-origin goods measure energy produced in Canada (foreign goods re-exported through Canada are excluded); "
    "one trade measure is kept so nothing is double counted; other products, partners and months are outside the research question")
exp[["month", "product_code", "product_name", "destination", "value"]].head()

Country of origin kept: Canada
[Exports] Series selection: 768 raw rows reduced to 768 rows; trade types dropped: none


,month,product_code,product_name,destination,value
0,2018-01-01,C12,Energy products,United States,10020138
1,2018-02-01,C12,Energy products,United States,7772901
2,2018-03-01,C12,Energy products,United States,8436578
3,2018-04-01,C12,Energy products,United States,8343371
4,2018-05-01,C12,Energy products,United States,9491938


#### 2.1.3 Units

In [7]:
scalar = sorted(exp["SCALAR_FACTOR"].dropna().astype(str).unique()) if "SCALAR_FACTOR" in exp else ["(column not found)"]
uom = sorted(exp["UOM"].dropna().astype(str).unique()) if "UOM" in exp else ["(column not found)"]
log("Exports", "Units",
    f"Unit of measure: {uom}; scalar factor: {scalar}",
    "Kept values in thousands of Canadian dollars and named the column export_value_cad_k",
    "Keeps the published precision; the unit is recorded in the variable name and the data dictionary")

[Exports] Units: Unit of measure: ['Dollars']; scalar factor: ['thousands']


#### 2.1.4 Missing values and quality flags

In [8]:
SYMBOLS = {"..": "not available", "x": "suppressed (confidentiality)", "E": "use with caution",
           "F": "too unreliable to be published", "p": "preliminary", "r": "revised"}

n_missing = int(exp["value"].isna().sum())
flag_cols = [c for c in ["STATUS", "SYMBOL"] if c in exp.columns]
flags = {}
for c in flag_cols:
    vc = exp[c].dropna().astype(str).str.strip()
    vc = vc[vc != ""].value_counts()
    flags.update({f"{c}={k} ({SYMBOLS.get(k, 'see StatCan legend')})": int(v) for k, v in vc.items()})
print("Missing values:", n_missing)
print("Quality flags:", flags or "none")
if n_missing:
    display(exp.loc[exp["value"].isna(), ["month", "product_code", "destination", "VALUE"] + flag_cols])

log("Exports", "Missing values",
    f"{n_missing} missing export values",
    "None to fix" if n_missing == 0 else "Kept as missing (not filled with zero) and listed above",
    "A missing value means the figure was not published; replacing it with zero would understate exports")
log("Exports", "Quality flags (STATUS / SYMBOL)",
    f"{flags if flags else 'No flags'}",
    "Kept all values and carried the flag into the clean table" if flags else "No action needed",
    "Flagged values are still the official figures; the flag lets later steps treat them with caution")

Missing values: 0
Quality flags: none
[Exports] Missing values: 0 missing export values
[Exports] Quality flags (STATUS / SYMBOL): No flags


#### 2.1.5 Duplicates and completeness

In [9]:
key = ["month", "product_code", "destination"]
n_dup = int(exp.duplicated(subset=key).sum())
exp = exp.drop_duplicates(subset=key, keep="first")
log("Exports", "Duplicate rows",
    f"{n_dup} duplicate month-product-destination rows",
    "None to fix" if n_dup == 0 else "Removed duplicates, keeping the first row",
    "Each series must have exactly one value per month")

counts = exp.groupby(["product_code", "destination"])["month"].nunique()
incomplete = counts[counts != N_MONTHS]
print(counts.to_string())
log("Exports", "Completeness",
    f"{len(counts)} series; {len(counts) - len(incomplete)} have all {N_MONTHS} months" +
    (f"; incomplete: {incomplete.to_dict()}" if len(incomplete) else ""),
    "No gaps to fix" if len(incomplete) == 0 else "Listed the incomplete series for review",
    "Month-over-month changes need an unbroken monthly series")

[Exports] Duplicate rows: 0 duplicate month-product-destination rows
product_code  destination  
141           United States    96
142           United States    96
143           United States    96
144           United States    96
145           United States    96
146           United States    96
261           United States    96
C12           United States    96
[Exports] Completeness: 8 series; 8 have all 96 months


#### 2.1.6 Consistency checks
- Exports to the U.S. should never be larger than exports to all countries.
- The six energy sub-groups (C121–C126) should add up to the energy total (C12).

In [10]:
wide = exp.pivot_table(index=["month", "product_code"], columns="destination", values="value", aggfunc="first")
if DEST_US in wide.columns and DEST_ALL not in wide.columns:
    print(f"'{DEST_ALL}' not downloaded: U.S. vs all-countries check skipped")
if DEST_US in wide.columns and DEST_ALL in wide.columns:
    n_bad = int((wide[DEST_US] > wide[DEST_ALL] + 0.5).sum())
    log("Exports", "U.S. vs all countries",
        f"{n_bad} cases where U.S. exports exceed exports to all countries",
        "None to fix" if n_bad == 0 else "Listed the cases for review",
        "The U.S. is one of the destinations included in 'all countries'")

subs = exp[exp["product_code"] != "C12"]                # the sub-groups of Energy products
total = exp[exp["product_code"] == "C12"]
if len(subs) and len(total):
    s = subs.groupby(["month", "destination"])["value"].sum(min_count=1)
    t = total.set_index(["month", "destination"])["value"]
    gap = ((s - t).abs() / t).dropna()
    print(f"Largest gap between sum of sub-groups and C12 total: {gap.max():.4%}")
    log("Exports", "Sub-groups add up to total",
        f"{subs['product_code'].nunique()} sub-groups; largest gap vs C12 total: {gap.max():.3%}",
        "Accepted (gap caused by rounding only)" if gap.max() < 0.01 else "Gaps above 1% listed for review",
        "Confirms the product categories are defined consistently across the whole period")
else:
    print("Sub-groups not downloaded: check skipped")

'All countries' not downloaded: U.S. vs all-countries check skipped
Largest gap between sum of sub-groups and C12 total: 0.0000%
[Exports] Sub-groups add up to total: 7 sub-groups; largest gap vs C12 total: 0.000%


#### 2.1.7 Unusual month-over-month changes (flagged, not removed)

In [11]:
us_total = total[total["destination"] == DEST_US].sort_values("month").set_index("month")["value"]
pct = us_total.pct_change()
big = pct[pct.abs() > EXPORT_OUTLIER_PCT]
print(big.map("{:+.1%}".format).to_string() if len(big) else "No large changes")
log("Exports", "Unusual changes (energy exports to U.S.)",
    f"{len(big)} months with a change larger than ±{EXPORT_OUTLIER_PCT:.0%}: " +
    (", ".join(f"{d:%Y-%m} ({v:+.0%})" for d, v in big.items()) if len(big) else "none"),
    "Kept all values and flagged these months",
    "Checked against known events (for example the 2020 oil-price collapse); genuine market movements are kept so the analysis reflects real volatility")

if "TERMINATED" in exp.columns:
    n_term = int(exp["TERMINATED"].notna().sum() - (exp["TERMINATED"].astype(str).str.strip() == "").sum())
    log("Exports", "Terminated series", f"{n_term} rows from terminated series",
        "None to fix" if n_term == 0 else "Listed for review", "A terminated series would stop being updated")

month
2019-03-01    +33.9%
2020-04-01    -37.0%
2020-05-01    -26.7%
2020-07-01    +49.3%
2022-03-01    +25.0%
2025-12-01    +26.4%
[Exports] Unusual changes (energy exports to U.S.): 6 months with a change larger than ±25%: 2019-03 (+34%), 2020-04 (-37%), 2020-05 (-27%), 2020-07 (+49%), 2022-03 (+25%), 2025-12 (+26%)
[Exports] Terminated series: 0 rows from terminated series


In [12]:
exports_clean = (exp.assign(status=exp[flag_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip() if flag_cols else "")
                   .rename(columns={"value": "export_value_cad_k"})
                   [["month", "product_code", "product_name", "destination", "export_value_cad_k", "status"]]
                   .sort_values(["product_code", "destination", "month"])
                   .reset_index(drop=True))
print(exports_clean.shape)
exports_clean.head(10)

(768, 6)


,month,product_code,product_name,destination,export_value_cad_k,status
0,2018-01-01,141,Crude oil and bitumen,United States,7096708,
1,2018-02-01,141,Crude oil and bitumen,United States,5755592,
2,2018-03-01,141,Crude oil and bitumen,United States,6342070,
3,2018-04-01,141,Crude oil and bitumen,United States,6519028,
4,2018-05-01,141,Crude oil and bitumen,United States,7782490,
5,2018-06-01,141,Crude oil and bitumen,United States,7965624,
6,2018-07-01,141,Crude oil and bitumen,United States,8168446,
7,2018-08-01,141,Crude oil and bitumen,United States,8411045,
8,2018-09-01,141,Crude oil and bitumen,United States,6992975,
9,2018-10-01,141,Crude oil and bitumen,United States,7691990,


### 2.2 Exchange rate – Bank of Canada daily FXUSDCAD

In [13]:
def read_valet_csv(path):
    """Bank of Canada Valet CSVs start with notes; the data begins after the line "OBSERVATIONS"."""
    text = path.read_text(encoding="utf-8-sig")
    lines = text.splitlines()
    start = next((i for i, l in enumerate(lines) if l.strip().strip('"') == "OBSERVATIONS"), None)
    body = "\n".join(lines[start + 1:]) if start is not None else text
    return pd.read_csv(io.StringIO(body), dtype=str), start is not None

if SOURCE == "csv":
    fx_path = RAW_DIR / FX_CSV if FX_CSV else find_file(["*FXUSDCAD*.csv", "*fx*.csv", "*FX*.csv"])
    raw_fx, had_notes = read_valet_csv(fx_path)
    print("Read file:", fx_path.name)
else:
    raw_fx, had_notes = md_con().sql(f"SELECT * FROM {RAW_FX_TABLE}").df(), False
    print("Read MotherDuck table:", RAW_FX_TABLE)

raw_fx.columns = [c.strip() for c in raw_fx.columns]
fx_date_col = find_col(raw_fx, "date")
fx_val_col = "FXUSDCAD" if "FXUSDCAD" in raw_fx.columns else [c for c in raw_fx.columns if c != fx_date_col][0]
print("Date column:", fx_date_col, "| Rate column:", fx_val_col, "| Rows:", len(raw_fx))
if had_notes:
    log("Exchange rate", "File format", "The CSV starts with notes (terms and series description) before the data",
        "Read only the rows after the 'OBSERVATIONS' line", "The notes are not data and would break the table structure")
raw_fx.head()

Read file: FXUSDCAD.csv
Date column: date | Rate column: FXUSDCAD | Rows: 1995
[Exchange rate] File format: The CSV starts with notes (terms and series description) before the data


,date,FXUSDCAD
0,2018-01-02,1.2517
1,2018-01-03,1.2533
2,2018-01-04,1.2515
3,2018-01-05,1.2403
4,2018-01-08,1.2422


In [14]:
fx = pd.DataFrame({"date": pd.to_datetime(raw_fx[fx_date_col], errors="coerce"),
                   "usd_cad": pd.to_numeric(raw_fx[fx_val_col], errors="coerce")})
log("Exchange rate", "Data types",
    f"Date and rate were read as {raw_fx[fx_date_col].dtype} / {raw_fx[fx_val_col].dtype}; {int(fx['date'].isna().sum())} dates could not be parsed",
    "Converted the date to a date type and the rate to a number",
    "Needed to calculate monthly averages and join with the export data")

n_before = len(fx)
fx = fx[fx["date"].between(PERIOD_START, PERIOD_END)].copy()
log("Exchange rate", "Study period",
    f"{n_before} rows read; {len(fx)} in {START_MONTH} to {END_MONTH} (first {fx['date'].min():%Y-%m-%d}, last {fx['date'].max():%Y-%m-%d})",
    "Kept only dates inside the study period", "Matches the period of the export data")

n_missing_fx = int(fx["usd_cad"].isna().sum())
fx = fx.dropna(subset=["usd_cad"])
log("Exchange rate", "Missing values",
    f"{n_missing_fx} listed dates without a rate",
    "None to fix" if n_missing_fx == 0 else "Removed these rows (no value was published)",
    "Monthly averages should use only published rates; a blank must not count as zero")

n_dup_fx = int(fx.duplicated(subset="date").sum())
fx = fx.drop_duplicates(subset="date", keep="first").sort_values("date").reset_index(drop=True)
log("Exchange rate", "Duplicate dates", f"{n_dup_fx} duplicate dates",
    "None to fix" if n_dup_fx == 0 else "Removed duplicates, keeping the first", "One rate per business day")

[Exchange rate] Data types: Date and rate were read as str / str; 0 dates could not be parsed
[Exchange rate] Study period: 1995 rows read; 1995 in 2018-01 to 2025-12 (first 2018-01-02, last 2025-12-31)
[Exchange rate] Missing values: 0 listed dates without a rate
[Exchange rate] Duplicate dates: 0 duplicate dates


In [15]:
n_weekend = int((fx["date"].dt.dayofweek >= 5).sum())
days_per_month = fx.groupby(fx["date"].dt.to_period("M")).size()
missing_months = sorted(set(pd.period_range(START_MONTH, END_MONTH, freq="M")) - set(days_per_month.index))
log("Exchange rate", "Weekends, holidays and coverage",
    f"{n_weekend} weekend dates; {len(days_per_month)} of {N_MONTHS} months covered; "
    f"{days_per_month.min()}–{days_per_month.max()} business days per month" +
    (f"; months missing: {[str(m) for m in missing_months]}" if missing_months else ""),
    "Weekends and holidays not treated as missing; monthly averages will use business days only",
    "The Bank of Canada does not publish rates on weekends and statutory holidays")

print(f"Rate range: {fx['usd_cad'].min():.4f} to {fx['usd_cad'].max():.4f} CAD per USD")
n_nonpos = int((fx["usd_cad"] <= 0).sum())
jumps = fx.set_index("date")["usd_cad"].pct_change()
jumps = jumps[jumps.abs() > FX_JUMP_PCT]
log("Exchange rate", "Valid range and sudden jumps",
    f"Range {fx['usd_cad'].min():.4f}–{fx['usd_cad'].max():.4f}; {n_nonpos} zero/negative values; "
    f"{len(jumps)} daily moves larger than ±{FX_JUMP_PCT:.0%}" +
    (": " + ", ".join(f"{d:%Y-%m-%d} ({v:+.1%})" for d, v in jumps.items()) if len(jumps) else ""),
    "Kept all values; large moves flagged",
    "All values are plausible exchange rates; large moves reflect real market events")
log("Exchange rate", "Unit and direction", "Rate is Canadian dollars per 1 U.S. dollar",
    "Named the column usd_cad and recorded the unit",
    "A higher value means a weaker Canadian dollar; this must be clear when interpreting results")

fx_daily_clean = fx
fx_daily_clean.head()

[Exchange rate] Weekends, holidays and coverage: 0 weekend dates; 96 of 96 months covered; 19–23 business days per month
Rate range: 1.2040 to 1.4603 CAD per USD
[Exchange rate] Valid range and sudden jumps: Range 1.2040–1.4603; 0 zero/negative values; 1 daily moves larger than ±2%: 2020-03-18 (+2.3%)
[Exchange rate] Unit and direction: Rate is Canadian dollars per 1 U.S. dollar


C:\Users\chenr\AppData\Local\Temp\ipykernel_4564\2566171535.py:1: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  n_weekend = int((fx["date"].dt.dayofweek >= 5).sum())


,date,usd_cad
0,2018-01-02,1.2517
1,2018-01-03,1.2533
2,2018-01-04,1.2515
3,2018-01-05,1.2403
4,2018-01-08,1.2422


### 2.3 Save the clean data and the cleaning log

In [16]:
quality_log = pd.DataFrame(LOG)
exports_clean.to_csv(CLEAN_DIR / "exports_clean.csv", index=False)
fx_daily_clean.to_csv(CLEAN_DIR / "fx_daily_clean.csv", index=False)
quality_log.to_csv(CLEAN_DIR / "quality_log.csv", index=False)

def to_md_table(df):
    esc = lambda x: str(x).replace("|", "\\|").replace("\n", " ")
    head = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "".join("| " + " | ".join(esc(v) for v in row) + " |\n" for row in df.itertuples(index=False))

(DOCS_DIR / "cleaning_log.md").write_text(
    "# Cleaning log\n\nGenerated by Part 2 of `notebooks/milestone2_pipeline.ipynb`. "
    f"Study period: {START_MONTH} to {END_MONTH}.\n\n" + to_md_table(quality_log), encoding="utf-8")

if SAVE_TO_MOTHERDUCK:
    con = md_con()
    con.execute("CREATE SCHEMA IF NOT EXISTS clean")
    for name, df in [("exports", exports_clean), ("fx_daily", fx_daily_clean), ("quality_log", quality_log)]:
        con.register("tmp_df", df)
        con.execute(f"CREATE OR REPLACE TABLE clean.{name} AS SELECT * FROM tmp_df")
        con.unregister("tmp_df")
    print("Saved to MotherDuck:", MD_DATABASE, "(clean.exports, clean.fx_daily, clean.quality_log)")

print("Saved: data/clean/exports_clean.csv, data/clean/fx_daily_clean.csv, data/clean/quality_log.csv, docs/cleaning_log.md")
quality_log

Saved: data/clean/exports_clean.csv, data/clean/fx_daily_clean.csv, data/clean/quality_log.csv, docs/cleaning_log.md


,Dataset,Check,Finding,Decision,Rationale
0,Exports,Data types,REF_DATE and VALUE were read as str / str; 0 d...,Converted REF_DATE to a monthly date (first da...,Dates and numbers need the correct type for so...
1,Exports,Series selection,768 raw rows reduced to 768 rows; trade types ...,Kept exports (Trade = 'Export') of Energy prod...,Canadian-origin goods measure energy produced ...
2,Exports,Units,Unit of measure: ['Dollars']; scalar factor: [...,Kept values in thousands of Canadian dollars a...,Keeps the published precision; the unit is rec...
3,Exports,Missing values,0 missing export values,None to fix,A missing value means the figure was not publi...
4,Exports,Quality flags (STATUS / SYMBOL),No flags,No action needed,Flagged values are still the official figures;...
5,Exports,Duplicate rows,0 duplicate month-product-destination rows,None to fix,Each series must have exactly one value per month
6,Exports,Completeness,8 series; 8 have all 96 months,No gaps to fix,Month-over-month changes need an unbroken mont...
7,Exports,Sub-groups add up to total,7 sub-groups; largest gap vs C12 total: 0.000%,Accepted (gap caused by rounding only),Confirms the product categories are defined co...
8,Exports,Unusual changes (energy exports to U.S.),6 months with a change larger than ±25%: 2019-...,Kept all values and flagged these months,Checked against known events (for example the ...
9,Exports,Terminated series,0 rows from terminated series,None to fix,A terminated series would stop being updated


#### Hand-off to Part 3

Part 2 produces two clean tables that later parts can use directly (they are also saved in `data/clean/`):

| Variable | Rows | Columns | Notes |
|---|---|---|---|
| `exports_clean` | 768 (8 series × 96 months) | `month`, `product_code`, `product_name`, `destination`, `export_value_cad_k`, `status` | Long format; `product_code` = `C12` is the energy total, the other 7 codes are sub-groups; values in thousands of CAD |
| `fx_daily_clean` | 1,995 business days | `date`, `usd_cad` | Canadian dollars per 1 U.S. dollar; no weekends or holidays |

---
## Part 3 – Data transformation and integration 

_Add your code below. Inputs: `exports_clean` and `fx_daily_clean` from Part 2._

---
## Part 4 – Prepared dataset and data storage 